# 🛡️ Ortho-FastDTR: Interactive Gradio Dashboard & Live GPU Sandbox

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pranay7427/Ortho-FastDTR/blob/main/Ortho_FastDTR_Colab.ipynb)

This notebook runs `app.py` and live GPU defense on a **Google Colab GPU** (Free T4 or A100/L4).

### ⚠️ Step 0: Ensure GPU Runtime is Enabled
1. Go to top menu: **Runtime** → **Change runtime type**
2. Under **Hardware accelerator**, select **T4 GPU** (or A100 / L4 if available)
3. Click **Save**

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Please change runtime to T4 GPU under Runtime -> Change runtime type.")

### Step 2: Clone the Ortho-FastDTR Repository

In [ ]:
import os
if not os.path.exists("/content/Ortho-FastDTR") and not os.path.exists("app.py"):
    !git clone https://github.com/Pranay7427/Ortho-FastDTR.git
    %cd Ortho-FastDTR
elif os.path.exists("/content/Ortho-FastDTR"):
    %cd /content/Ortho-FastDTR

### Step 3: Install Required Dependencies
Installs core VLM dependencies (`transformers==4.31.0`, `accelerate`, `gradio`, and the local `llava` package).

In [ ]:
!pip install -r requirements.txt

### Option A: Launch Interactive 4-Tab Web App (Gradio)
Launches `app.py` with `share=True`, generating a public **`https://xxxx.gradio.live`** URL to test Tab 4 (Live GPU Sandbox) in your browser.

In [ ]:
!python app.py

### Option B: Direct In-Notebook Live GPU Test (No Browser UI Needed)
Execute live GPU defense directly in this notebook cell: loads `LLaVA-Llama-2-7B` + Rank-4 SVD Subspace, evaluates sample #1, plots the side-by-side $16\times 16$ patch eviction heatmaps, and prints both model responses inline.

In [ ]:
import torch
import matplotlib.pyplot as plt
import app

# 1. Load Live Model + Rank-4 Refusal Subspace onto GPU
print("Loading LLaVA-Llama-2-7B and computing Rank-4 SVD subspace on GPU...")
status = app.load_live_model()
print(status)

# 2. Load HADES Test Sample #1
img, prompt, sample_status = app.load_hades_sample_into_sandbox(1)
print(sample_status)
print("Evaluated Prompt:", prompt)

# 3. Execute Live Defense (Vanilla DTR vs. Ortho-FastTaylor)
print("\nRunning real-time defense attribution on GPU...")
van_map, fast_map, van_txt, fast_txt = app.run_live_defense(img, prompt, evict_quantile=0.65, sample_idx=1)

# 4. Render Side-by-Side Eviction Heatmaps Inline
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img)
axes[0].set_title("Input Image (HADES S+T+A #1)", fontsize=13)
axes[0].axis("off")

axes[1].imshow(van_map)
axes[1].set_title("Vanilla DTR Heatmap (10-Step 1D)", fontsize=13)
axes[1].axis("off")

axes[2].imshow(fast_map)
axes[2].set_title("Ortho-FastTaylor Heatmap (Ours: 1-Step Subspace)", fontsize=13)
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("="*70)
print(van_txt)
print("="*70)
print(fast_txt)
print("="*70)